In [0]:
# Define your file path
file_path = (
    "/Volumes/workspace/default/movies_landing/movies_complete_dataset.parquet"
)

# Read the Parquet file into PySpark
df_bronze = spark.read.format("parquet").load(file_path)

# Preview the data and check row count
display(df_bronze.limit(10))
print(f"Total rows in Bronze: {df_bronze.count()}")

movie_id,title,release_year,runtime,rating,vote_count,genres,countries,director,plot,poster,data_source,budget,revenue,profit,roi,daily_box_office,box_office_per_runtime,decade,popularity_score
imdb_0,Unknown,null,null,null,null,Action|Adventure|Fantasy|Sci-Fi,USA,,,,imdb,1.1267476559478506E8,8.811125559895533E8,7.684377903947682E8,6.819963514797265,2.937041853298511E7,null,null,null
1,Toy Story (1995),null,null,null,null,Adventure|Animation|Children|Comedy|Fantasy,,,,,movielens,2.852389347697699E8,9.413629552433238E8,6.561240204735539E8,2.300261081128855,3.137876517477746E7,null,null,null
2,Jumanji (1995),null,null,null,null,Adventure|Children|Fantasy,,,,,movielens,2.1973218557251582E8,9.737571233906358E8,7.5402493781812E8,3.4315634546368146,3.2458570779687863E7,null,null,null
3,Grumpier Old Men (1995),null,null,null,null,Comedy|Romance,,,,,movielens,1.7979821601701248E8,2.2504660273151116E9,2.070667811298099E9,11.516620449127108,7.501553424383706E7,null,null,null
4,Waiting to Exhale (1995),null,null,null,null,Comedy|Drama|Romance,,,,,movielens,4.722758281250974E7,4.901918906359963E8,4.4296430782348657E8,9.379355906950066,1.6339729687866542E7,null,null,null
5,Father of the Bride Part II (1995),null,null,null,null,Comedy,,,,,movielens,4.722035884069269E7,5.185561795695291E8,4.713358207288364E8,9.981623018134655,1.728520598565097E7,null,null,null
6,Heat (1995),null,null,null,null,Action|Crime|Thriller,,,,,movielens,1.7896041844375737E7,2.1732104977253422E7,3836063.132877685,0.21435260189019173,724403.4992417807,null,null,null
7,Sabrina (1995),null,null,null,null,Comedy|Romance,,,,,movielens,2.5991975565959308E8,2.855373706203564E9,2.595453950543971E9,9.985597069978542,9.51791235401188E7,null,null,null
8,Tom and Huck (1995),null,null,null,null,Adventure|Children,,,,,movielens,1.8053394601709104E8,1.3490985878806446E9,1.1685646418635535E9,6.472825015152144,4.496995292935482E7,null,null,null
9,Sudden Death (1995),null,null,null,null,Action,,,,,movielens,2.125677370499156E8,1.4869810712219117E9,1.274413334171996E9,5.995328133322206,4.956603570739706E7,null,null,null


Total rows in Bronze: 78684


In [0]:
# 2. Save the raw data as a Bronze Delta Table
spark.sql("CREATE SCHEMA IF NOT EXISTS movies_bronze")

table_name = "movies_bronze.movies"

df_bronze.write.format("delta").mode("overwrite").saveAsTable(table_name)

print(f"Bronze table {table_name} successfully created!")

Bronze table movies_bronze.movies successfully created!


In [0]:
from pyspark.sql.functions import col, regexp_extract, trim, translate

# 1. Read the raw data from Bronze Delta table
df_bronze = spark.read.table("movies_bronze.movies")

# Optional: Print columns to verify their exact names if needed
print("Original Columns:", df_bronze.columns)

# 2. Apply Silver Layer Cleaning & Transformations
df_silver = (
    df_bronze.dropDuplicates()  # Step A: Remove exact duplicate rows
    # Step B: Trim accidental whitespace
    .withColumn("title", trim(col("title")))
    # Step C: Handle Nulls - fill missing numerical metrics with 0 or default values
    .na.fill({"budget": 0.0, "revenue": 0.0, "vote_count": 0.0})
)

# Step D: Type Casting
df_silver = df_silver \
    .withColumn("budget", col("budget").cast("double")) \
    .withColumn("revenue", col("revenue").cast("double")) \
    .withColumn("vote_count", col("vote_count").cast("float")) \
    .withColumn("popularity_score", col("popularity_score").cast("float")) \
    .withColumn("genres", translate("genres", "|", ","))

# 3. Save the cleaned dataset as the Silver Delta Table
spark.sql("CREATE SCHEMA IF NOT EXISTS movies_silver")

silver_table_name = "movies_silver.movies"

df_silver.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)

print(f"Silver table '{silver_table_name}' successfully created and cleaned!")

# Preview your clean silver data
display(df_silver.limit(10))

Original Columns: ['movie_id', 'title', 'release_year', 'runtime', 'rating', 'vote_count', 'genres', 'countries', 'director', 'plot', 'poster', 'data_source', 'budget', 'revenue', 'profit', 'roi', 'daily_box_office', 'box_office_per_runtime', 'decade', 'popularity_score']
Silver table 'movies_silver.movies' successfully created and cleaned!


movie_id,title,release_year,runtime,rating,vote_count,genres,countries,director,plot,poster,data_source,budget,revenue,profit,roi,daily_box_office,box_office_per_runtime,decade,popularity_score
imdb_0,Unknown,null,null,null,0.0,"Action,Adventure,Fantasy,Sci-Fi",USA,,,,imdb,1.1267476559478506E8,8.811125559895533E8,7.684377903947682E8,6.819963514797265,2.937041853298511E7,null,null,null
1,Toy Story (1995),null,null,null,0.0,"Adventure,Animation,Children,Comedy,Fantasy",,,,,movielens,2.852389347697699E8,9.413629552433238E8,6.561240204735539E8,2.300261081128855,3.137876517477746E7,null,null,null
2,Jumanji (1995),null,null,null,0.0,"Adventure,Children,Fantasy",,,,,movielens,2.1973218557251582E8,9.737571233906358E8,7.5402493781812E8,3.4315634546368146,3.2458570779687863E7,null,null,null
3,Grumpier Old Men (1995),null,null,null,0.0,"Comedy,Romance",,,,,movielens,1.7979821601701248E8,2.2504660273151116E9,2.070667811298099E9,11.516620449127108,7.501553424383706E7,null,null,null
4,Waiting to Exhale (1995),null,null,null,0.0,"Comedy,Drama,Romance",,,,,movielens,4.722758281250974E7,4.901918906359963E8,4.4296430782348657E8,9.379355906950066,1.6339729687866542E7,null,null,null
5,Father of the Bride Part II (1995),null,null,null,0.0,Comedy,,,,,movielens,4.722035884069269E7,5.185561795695291E8,4.713358207288364E8,9.981623018134655,1.728520598565097E7,null,null,null
6,Heat (1995),null,null,null,0.0,"Action,Crime,Thriller",,,,,movielens,1.7896041844375737E7,2.1732104977253422E7,3836063.132877685,0.21435260189019173,724403.4992417807,null,null,null
7,Sabrina (1995),null,null,null,0.0,"Comedy,Romance",,,,,movielens,2.5991975565959308E8,2.855373706203564E9,2.595453950543971E9,9.985597069978542,9.51791235401188E7,null,null,null
8,Tom and Huck (1995),null,null,null,0.0,"Adventure,Children",,,,,movielens,1.8053394601709104E8,1.3490985878806446E9,1.1685646418635535E9,6.472825015152144,4.496995292935482E7,null,null,null
9,Sudden Death (1995),null,null,null,0.0,Action,,,,,movielens,2.125677370499156E8,1.4869810712219117E9,1.274413334171996E9,5.995328133322206,4.956603570739706E7,null,null,null


In [0]:
from pyspark.sql.functions import col, expr, round

# 1. Read the clean data from Silver Delta table
df_silver = spark.read.table("movies_silver.movies")

# 2. Gold Layer Feature Engineering & Analytics Modeling
# Let's create high-value business metrics: Profit and Return on Investment (ROI)
df_gold = (
    df_silver
    # Calculate Net Profit (Revenue minus Budget)
    .withColumn("profit", col("revenue") - col("budget"))
    # Calculate ROI Percentage/Ratio safely (avoiding division by zero)
    .withColumn(
        "roi",
        round(
            expr(
                "CASE WHEN budget > 0 THEN (revenue - budget) / budget ELSE 0"
                " END"
            ),
            2,
        ),
    )
)

# 3. Save as the Gold Delta Table (Optimized for Power BI reporting)
spark.sql("CREATE SCHEMA IF NOT EXISTS movies_gold")

gold_table_name = "movies_gold.fact_movies"

df_gold.write.format("delta").mode("overwrite").saveAsTable(gold_table_name)

print(f"Gold table '{gold_table_name}' successfully created!")

# Preview the new financial metrics
display(
    df_gold.select(
        "title", "budget", "revenue", "profit", "roi", "genres"
    ).limit(10)
)

Gold table 'workspace.gold.fact_movies' successfully created!


title,budget,revenue,profit,roi,genres
Heat (1995),1.7896041844375737E7,2.1732104977253422E7,3836063.132877685,0.21,"Action,Crime,Thriller"
Tom and Huck (1995),1.8053394601709104E8,1.3490985878806446E9,1.1685646418635535E9,6.47,"Adventure,Children"
Nixon (1995),5.495657767852663E7,7.660503415506101E8,7.110937638720834E8,12.94,Drama
Mr. Holland's Opus (1995),2.4870688399100283E8,5.226409991450715E8,2.739341151540687E8,1.1,Drama
Nico Icon (1995),2.2676373194360062E7,2.3020765449427906E7,344392.25506784394,0.02,Documentary
"Amazing Panda Adventure, The (1995)",1.1547640732682573E7,1.3370994797532186E8,1.2216230724263929E8,10.58,"Adventure,Children"
"Awfully Big Adventure, An (1995)",1.5105236745704404E8,6.191125037906084E8,4.680601363335644E8,3.1,Drama
Casper (1995),2.286050747909509E8,1.2360029761124096E9,1.0073979013214587E9,4.41,"Adventure,Children"
Desperado (1995),1.9024214837288794E8,2.0877129696767726E9,1.8974708213038847E9,9.97,"Action,Romance,Western"
Clerks (1994),1.7070842669897375E8,1.9614029239141605E9,1.7906944972151868E9,10.49,Comedy


In [0]:
from pyspark.sql.functions import col, explode, trim, split, monotonically_increasing_id

# 1. Read from Silver table
df_silver = spark.read.table("movies_silver.movies")

# 2. Split the comma-separated genres, explode them into separate rows, and clean up
df_genres = (
    df_silver
    # Split string by comma into an array
    .withColumn("genre_array", split(col("genres"), ","))
    # Explode the array so each genre gets its own row
    .withColumn("genre", explode(col("genre_array")))
    # Trim any accidental whitespace
    .withColumn("genre", trim(col("genre")))
    # Select only the distinct genres
    .select("genre")
    .distinct()
    # Filter out empty or null genres
    .filter((col("genre").isNotNull()) & (col("genre") != ""))
    # Add a unique surrogate key for the dimension table
    .withColumn("genre_id", monotonically_increasing_id())
)

# 3. Save as a Gold Dimension Table
spark.sql("CREATE SCHEMA IF NOT EXISTS movies_gold")

dim_genre_table = "movies_gold.dim_genre"
df_genres.write.format("delta").mode("overwrite").saveAsTable(dim_genre_table)

print(f"Dimension table '{dim_genre_table}' successfully created!")

# Preview the unique genres
display(df_genres.limit(10))

Dimension table 'workspace.gold.dim_genre' successfully created!


genre,genre_id
Horror,0
Western,1
Romance,2
Comedy,3
Documentary,4
Children,5
Fantasy,6
IMAX,7
Thriller,8
Action,9


In [0]:
from pyspark.sql.functions import col, trim, monotonically_increasing_id

# 1. Read from Silver table
df_silver = spark.read.table("movies_silver.movies")

# 2. Extract distinct countries from the comma-separated 'countries' column
df_countries = (
    df_silver
    # Filter out rows with null or empty countries
    .filter((col("countries").isNotNull()) & (col("countries") != ""))
    # Select distinct countries directly (values are single, not comma-separated in practice)
    .select("countries")
    .distinct()
    # Clean up whitespace
    .withColumn("country", trim(col("countries")))
    .select("country")
    .distinct()
    .filter((col("country").isNotNull()) & (col("country") != ""))
    # Add a unique surrogate key
    .withColumn("country_id", monotonically_increasing_id())
)

# 3. Save as a Gold Dimension Table
spark.sql("CREATE SCHEMA IF NOT EXISTS movies_gold")

dim_country_table = "movies_gold.dim_country"
df_countries.write.format("delta").mode("overwrite").saveAsTable(dim_country_table)

print(f"Dimension table '{dim_country_table}' successfully created!")

# Preview the unique countries
display(df_countries.limit(10))

Dimension table 'workspace.gold.dim_country' successfully created!


country,country_id
UK,0
USA,1
France,2
Japan,3
South Korea,4
Germany,5
USSR,6
Denmark,7
Sweden,8
Canada,9


In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id

# 1. Read from Silver table
df_silver = spark.read.table("movies_silver.movies")

# 2. Extract distinct data sources
df_sources = (
    df_silver
    .select("data_source")
    .distinct()
    .filter((col("data_source").isNotNull()) & (col("data_source") != ""))
    .withColumn("source_id", monotonically_increasing_id())
)

# 3. Save as a Gold Dimension Table
spark.sql("CREATE SCHEMA IF NOT EXISTS movies_gold")

dim_source_table = "movies_gold.dim_data_source"
df_sources.write.format("delta").mode("overwrite").saveAsTable(dim_source_table)

print(f"Dimension table '{dim_source_table}' successfully created!")

# Preview the data sources
display(df_sources)

Dimension table 'workspace.gold.dim_data_source' successfully created!


data_source,source_id
movielens,0
letterboxd,1
wikipedia,2
imdb,3


In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id

# 1. Read from Silver table
df_silver = spark.read.table("movies_silver.movies")

# 2. Build a Time Dimension from release_year and decade
df_time = (
    df_silver
    .select("release_year", "decade")
    .distinct()
    .filter(col("release_year").isNotNull())
    .withColumn("year", col("release_year").cast("int"))
    .withColumn("decade", col("decade").cast("int"))
    .withColumn("century", (col("release_year") / 100).cast("int") + 1)
    .select("year", "decade", "century")
    .orderBy("year")
    .withColumn("year_id", monotonically_increasing_id())
)

# 3. Save as a Gold Dimension Table
spark.sql("CREATE SCHEMA IF NOT EXISTS movies_gold")

dim_time_table = "movies_gold.dim_time"
df_time.write.format("delta").mode("overwrite").saveAsTable(dim_time_table)

print(f"Dimension table '{dim_time_table}' successfully created!")

# Preview the time dimension
display(df_time.limit(10))

Dimension table 'workspace.gold.dim_time' successfully created!


year,decade,century,year_id
1888,1880,19,0
1892,1890,19,1
1895,1890,19,2
1896,1890,19,3
1898,1890,19,4
1899,1890,19,5
1900,1900,20,6
1901,1900,20,7
1902,1900,20,8
1903,1900,20,9
